In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

In [ ]:
# Task 2: Write your code here:
df_food.head()

In [ ]:
# Task 3: Write your code here:
df_food.info()

In [ ]:
# Task 4: Write your code here:
df_food.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df_food, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df_food.drop(columns=['Order_ID'])

In [ ]:
#df_food.shape()

In [ ]:
missing_percentage = (df_food.isnull().sum() / len(df_food)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage',ascending=False)

print("Missing Data Analysis:")
missing_data.head(20)

In [ ]:
# Task 2: Write your code here:

print("Missing values before cleaning:")
print(df_food.isnull().sum())
df_food['Delivery_Time'] = df_food['Delivery_Time'].fillna(df_food['Delivery_Time'].mean())
df_food['Weather']=df_food['Weather'].fillna(df_food['Weather'].mode()[0])
df_food['Traffic_Level']=df_food['Traffic_Level'].fillna(df_food['Traffic_Level'].mode()[0])
df_food['Time_of_Day']=df_food['Time_of_Day'].fillna(df_food['Time_of_Day'].mode()[0])
df_food['Courier_Experience_yrs']=df_food['Courier_Experience_yrs'].fillna(df_food['Courier_Experience_yrs'].mean())

print("Missing values after cleaning:")
print(df_food.isnull().sum())

In [ ]:
# Task 3: Write your code here:
duplicates = df_food.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates > 0:
  df_food.drop_duplicates(inplace=True)
  print(f"Removed {duplicates} duplicate rows")
else:
  print("No duplicates found!")

In [ ]:
# Task 4: Write your code here:

categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_food[col] = le.fit_transform(df_food[col].astype(str))

df_food.head()

In [ ]:
# Task 5: Write your code here:
numerical_cols = df_food.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_food[numerical_cols] = scaler.fit_transform(df_food[numerical_cols])
df_food.head()

In [ ]:
# Task 6: Write your code here:


def check_target_imbalance(df, target_column):
    print("Target Distribution:")
    print(df[target_column].value_counts(normalize=True))
    sns.countplot(x=df[target_column])
    plt.title("Target Distribution")
    print(f"Legendary: {df_food['Delivery_Time'].sum()}")
    print(f"Normal: {(df_food['Delivery_Time'] == 0).sum()}")
    plt.show()

check_target_imbalance(df_food, "Delivery_Time")


#imbalance

In [ ]:
# Task 1: Write your code here:
X = df_food.drop("Delivery_Time",axis=1)
y = df_food['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
model = RandomForestRegressor(n_estimators=200)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    # print shapes
    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)
    print("-" * 30)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)


print("MAE :", mae)


In [ ]:
# Task 1: Write your code here:
importance = model({
    'feature': features,
    'importance': model.feature_importances_
})
importance = importance.sort_values('importance', ascending=True).tail(10)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'])
plt.title('Top 10 Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: